# 01 - PortWatch dataset exploration

This notebook contains the first exploratory analysis of the PortWatch maritime chokepoints dataset. The goal is to understand the raw data before building features, models, and closure scenarios.

## 1. Load the dataset

The raw CSV is stored in `data/raw/`. Each row corresponds to one daily observation for one maritime chokepoint.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
raw_path = ROOT / 'data' / 'raw' / 'portwatch_daily_chokepoints.csv'
if not raw_path.exists():
    ROOT = Path('..').resolve()
    raw_path = ROOT / 'data' / 'raw' / 'portwatch_daily_chokepoints.csv'
df = pd.read_csv(raw_path, parse_dates=['date'])

df.shape

In [ ]:
df.head()

## 2. Basic structure

This section checks the available columns, the observation period, and the number of chokepoints in the dataset.

In [ ]:
pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
    'missing_values': df.isna().sum().values,
    'missing_rate': (df.isna().mean().values * 100).round(2),
})

In [ ]:
overview = {
    'start_date': df['date'].min(),
    'end_date': df['date'].max(),
    'rows': len(df),
    'chokepoints': df['portname'].nunique(),
}
overview

In [ ]:
sorted(df['portname'].dropna().unique())

## 3. Traffic by chokepoint

The first business question is: which chokepoints have the highest observed maritime traffic?

In [ ]:
traffic_summary = (
    df.groupby('portname')
    .agg(
        mean_total_ships=('n_total', 'mean'),
        mean_tankers=('n_tanker', 'mean'),
        mean_cargo=('n_cargo', 'mean'),
        observed_days=('date', 'nunique'),
    )
    .sort_values('mean_total_ships', ascending=False)
)
traffic_summary.head(10).round(2)

In [ ]:
top_10 = traffic_summary.head(10).sort_values('mean_total_ships')
ax = top_10['mean_total_ships'].plot(kind='barh', figsize=(9, 5), color='#2563eb')
ax.set_title('Top 10 chokepoints by average daily traffic')
ax.set_xlabel('Average ships per day')
ax.set_ylabel('Chokepoint')
plt.tight_layout()

## 4. Tanker exposure

Tankers are important for the business case because they are linked to energy flows. A chokepoint can be strategic even if its total traffic is not the highest.

In [ ]:
tanker_summary = traffic_summary.sort_values('mean_tankers', ascending=False)
tanker_summary[['mean_total_ships', 'mean_tankers', 'observed_days']].head(10).round(2)

In [ ]:
top_tankers = tanker_summary.head(10).sort_values('mean_tankers')
ax = top_tankers['mean_tankers'].plot(kind='barh', figsize=(9, 5), color='#0f766e')
ax.set_title('Top 10 chokepoints by average daily tanker traffic')
ax.set_xlabel('Average tankers per day')
ax.set_ylabel('Chokepoint')
plt.tight_layout()

## 5. Focus on selected strategic chokepoints

The project will focus especially on the Strait of Hormuz, Malacca Strait, and Taiwan Strait because they represent different forms of maritime risk: energy exposure, global trade volume, and regional strategic tension.

In [ ]:
focus_ports = ['Strait of Hormuz', 'Malacca Strait', 'Taiwan Strait']
focus = df[df['portname'].isin(focus_ports)].copy()

focus.groupby('portname').agg(
    mean_total_ships=('n_total', 'mean'),
    mean_tankers=('n_tanker', 'mean'),
    min_total_ships=('n_total', 'min'),
    max_total_ships=('n_total', 'max'),
    observed_days=('date', 'nunique'),
).round(2)

In [ ]:
pivot_focus = focus.pivot_table(index='date', columns='portname', values='n_total', aggfunc='mean')
ax = pivot_focus.rolling(7).mean().plot(figsize=(11, 5), linewidth=2)
ax.set_title('7-day rolling average traffic for selected chokepoints')
ax.set_xlabel('Date')
ax.set_ylabel('Ships per day')
plt.tight_layout()

## First observations

- Malacca Strait has very high total traffic and tanker traffic, which makes it a major global chokepoint.
- The Strait of Hormuz has lower total traffic than Malacca, but a high tanker exposure, making it highly strategic for energy flows.
- Taiwan Strait and Korea Strait are among the most active routes in the dataset.
- The next step is to transform these observations into features that can support a criticality score and later modeling.